In [1]:
import pandas as pd

df = pd.read_csv("usb_merged_final_data.csv")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.columns.tolist())

(29775, 16)
id                              object
sessionStart                    object
sessionStop                     object
connectorType                   object
energy_consumption(kWh)        float64
authType                        object
authId                           int64
chargerId                        int64
locationId                       int64
duration_hrs                   float64
time_based_util_rate           float64
Month                            int64
Season                          object
day                             object
Carbon Intensity (gCO2/kWh)      int64
Carbon Emissions (gCO2)        float64
dtype: object
                                     id      sessionStart       sessionStop  \
0  1ff7bb21-86f6-4a58-9713-63c9551b0dc3  18/03/2021 17:39  18/03/2021 17:59   
1  4ec4bf07-f359-4199-93d3-ec842a450cfa  18/03/2021 19:00  18/03/2021 19:33   
2  63ef2126-e813-4225-982d-fcd89108f2ba  18/03/2021 20:43  18/03/2021 21:07   
3  707dbb44-cb45-4cea-ab86-dec7c06

In [2]:
df = pd.read_csv("usb_merged_final_data.csv")
df["sessionStart"] = pd.to_datetime(df["sessionStart"], dayfirst=True)
df["sessionStop"] = pd.to_datetime(df["sessionStop"], dayfirst=True)

print("Date range:", df["sessionStart"].min(), "→", df["sessionStart"].max())
print("Sessions per year:\n", df["sessionStart"].dt.year.value_counts().sort_index())

Date range: 2021-03-18 17:39:00 → 2024-07-01 15:20:00
Sessions per year:
 sessionStart
2021     4035
2022     8515
2023    11264
2024     5961
Name: count, dtype: int64


In [4]:
import duckdb

con = duckdb.connect("data/ev_twin.duckdb")

print("=== metering_raw ===")
print(con.execute("SELECT COUNT(*) FROM metering_raw").fetchone())
print(con.execute("SELECT * FROM metering_raw LIMIT 3").df())

print("=== charging_sessions ===")
print(con.execute("SELECT COUNT(*) FROM charging_sessions").fetchone())
print(con.execute("SELECT * FROM charging_sessions LIMIT 3").df())

print("=== checkpoint ===")
print(con.execute("SELECT status, COUNT(*) FROM scrape_checkpoint GROUP BY status").df())

con.close()

=== metering_raw ===
(28728144,)
      meter_id            datetime  value
0  GM/C/030/01 2021-03-18 00:30:00   9.73
1  GM/C/030/01 2021-03-18 05:00:00   9.73
2  GM/C/030/01 2021-03-18 14:00:00   9.73
=== charging_sessions ===
(29775,)
                                     id       session_start  \
0  1ff7bb21-86f6-4a58-9713-63c9551b0dc3 2021-03-18 17:39:00   
1  4ec4bf07-f359-4199-93d3-ec842a450cfa 2021-03-18 19:00:00   
2  63ef2126-e813-4225-982d-fcd89108f2ba 2021-03-18 20:43:00   

         session_stop      connector_type  energy_kwh auth_type  auth_id  \
0 2021-03-18 17:59:00  IEC_62196_T2_COMBO       9.509  CUSTOMER  1062232   
1 2021-03-18 19:33:00             CHADEMO       6.566  CUSTOMER  1056758   
2 2021-03-18 21:07:00             CHADEMO      13.882  CUSTOMER  1076569   

   charger_id  location_id  duration_hrs  time_based_util_rate  month  season  \
0     5000198        50112      0.325000              1.354167      3  Spring   
1     5000194        50112      0.557500    